# **STQD6324 DATA MANAGEMENT ASSIGNMENT 1 (P166974)**

## **DATA LOADING**
<p align="justify">
The Iris dataset contains 150 observations of iris flowers with four numerical features (Sepal length, Sepal width, Petal length, Petal width) and a categorical target variable representing three species (Setosa, Versicolor, Virginica). The first step is to load the dataset into a Spark data frame. The process is shown as per below code:
</p>

In [39]:
# Imports the SparkSession class from the pyspark.sql module
from pyspark.sql import SparkSession

# Imports the requests library for making HTTP requests
import requests

# Create a Spark session (entry point to PySpark)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Iris") \
    .getOrCreate() # Gets an existing SparkSession or creates a new one

# URL of the Iris dataset
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

# Define the local filename where the dataset will be saved
local_filename = "iris.csv"

# Downloads the CSV file from the URL
response = requests.get(url)

# Check if request was successful (raises error if status code is 4xx or 5xx)
response.raise_for_status()

# Opens the local file in binary write mode
with open(local_filename, 'wb') as f:
    f.write(response.content) # Writes the content of the downloaded file to the local file

# Now read the local file with Spark
df = spark.read.csv(local_filename, header=True, inferSchema=True) # Reads the local CSV into a Spark DataFrame
# header=True: Uses the first row as column names
# inferSchema=True: Automatically detects data types

# Displays the first 10 rows of the DataFrame
df.show(10)

# Prints the schema of the DataFrame
df.printSchema()

# Computes and displays summary statistics (count, mean, stddev, min, max)
df.describe().show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
|         5.4|        3.9|         1.7|        0.4| setosa|
|         4.6|        3.4|         1.4|        0.3| setosa|
|         5.0|        3.4|         1.5|        0.2| setosa|
|         4.4|        2.9|         1.4|        0.2| setosa|
|         4.9|        3.1|         1.5|        0.1| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 10 rows
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)

<p align="justify">
This code above demonstrates the complete data loading and initial exploration process using PySpark for the Iris dataset. First, the required libraries are imported: "SparkSession" from "pyspark.sql" to create a Spark environment, and "requests" to download the dataset from an online source. A Spark session is then created using "SparkSession.builder", which acts as the entry point for all PySpark operations. The configuration ".master("local[*]")" allows Spark to run locally using all available CPU cores, and ."appName("Iris")" assigns a name to the application.
</p>
<p align="justify">
Next, the Iris dataset is downloaded from a GitHub URL using the "requests" library. The response is checked using "raise_for_status()" to ensure the download is successful. The dataset is then saved locally as a CSV file named "iris.csv".
</p>
<p align="justify">
After downloading, the file is loaded into a Spark DataFrame using "spark.read.csv()". The parameters "header=True" and "inferSchema=True" ensure that column names are correctly assigned and data types are automatically detected.
</p>
<p align="justify">
Finally, the dataset is explored using several functions: "df.show(10)" displays the first 10 rows of the dataset, "df.printSchema()" shows the structure and data types of each column, and "df.describe().show()" provides summary statistics such as count, mean, standard deviation, minimum, and maximum values. This step helps in understanding the dataset before further preprocessing and model development.
</p>

## **DATA PREPROCESSING**
<p align="justify">
Two key preprocessing steps were performed to prepare the dataset for a classification task using PySpark. Firstly, label encoding was applied to the target variable species, which originally exists in string format (Setosa, Versicolor, Virginica). This was achieved using the "StringIndexer()", which converts each unique category into a corresponding numerical label. This step is necessary because machine learning algorithms in Spark cannot process categorical string values directly. Secondly, feature vectorization was carried out using the "VectorAssembler()", where all four numerical feature columns; sepal length, sepal width, petal length, and petal width were combined into a single vector column named "features". This transformation organizes the input variables into a structured format that can be efficiently processed by Spark Machine Learning (ML) models. These preprocessing steps are essential because Spark MLlib requires the input data to be represented as numerical labels and feature vectors in order to perform model training and prediction effectively. This is shown as per below code:
</p>

In [40]:
# Import StringIndexer to convert text labels to numerical indices
from pyspark.ml.feature import StringIndexer

# Initialize StringIndexer: specify the input column ('species') and the new output column ('label')
label_indexer = StringIndexer(inputCol="species", outputCol="label")

# Fit the indexer to the data to learn the mapping and then transform the DataFrame
df = label_indexer.fit(df).transform(df)

# Import VectorAssembler to combine multiple columns into a single vector column
from pyspark.ml.feature import VectorAssembler

# Initialize VectorAssembler with the list of input feature columns and the name of the output vector column
assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

# Transform the DataFrame to create the 'features' vector column
df = assembler.transform(df)

# Display the first 10 rows of the newly created features and label columns
df.select("features", "label").show(10)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|  0.0|
|[4.9,3.0,1.4,0.2]|  0.0|
|[4.7,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
|[5.0,3.6,1.4,0.2]|  0.0|
|[5.4,3.9,1.7,0.4]|  0.0|
|[4.6,3.4,1.4,0.3]|  0.0|
|[5.0,3.4,1.5,0.2]|  0.0|
|[4.4,2.9,1.4,0.2]|  0.0|
|[4.9,3.1,1.5,0.1]|  0.0|
+-----------------+-----+
only showing top 10 rows


<p align="justify">
After this transformation, the dataset is structured in the exact format required by PySpark’s machine learning library. All the independent variables; sepal length, sepal width, petal length, and petal width are combined into a single column called "features", which is stored as a vector representing the input data for the model. At the same time, the categorical variable species has been converted into a numeric column called "label", which represents the target variable that the model will learn to predict. This format is essential because Spark ML algorithms expect the input data to be organized with a single features vector and a corresponding label column, allowing the model to efficiently process the data during training and make predictions.
</p>

## **TRAIN TEST SPLIT**
<p align="justify">
The dataset was divided into two subsets consisting of 80% training data and 20% testing data. Given the relatively small size of the dataset (150 observations), an 80:20 train-test split was selected to balance model training and evaluation. This proportion provides sufficient data for training while retaining an adequate number of samples for testing. The training dataset is used to build and learn the patterns of the model, while the testing dataset is reserved for evaluating the model’s performance on unseen data. A fixed random seed was applied during the splitting process to ensure reproducibility, meaning that the same data partition can be obtained consistently across different runs. This step is essential because it enables a fair and unbiased assessment of the model’s generalization ability, ensuring that the evaluation reflects how well the model performs on new, unseen observations rather than just the data it was trained on. The process of splitting the data into train and test data is shown as per below code:
</p>

In [41]:
# train test split (80% train and 20% test)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

<p align="justify">
This line of code splits the dataset into two parts: 80% for training and 20% for testing using PySpark’s "randomSplit()" function. The training dataset "(train_df)" is used to build and train the machine learning model, while the testing dataset "(test_df)" is used to evaluate how well the model performs on unseen data. The parameter "seed=42" ensures that the split is reproducible, meaning the same data division will be obtained every time the code is run.
</p>

## **CLASSIFICATION MODELS (MODEL DEVELOPMENT)**
<p align="justify">
Three classification algorithms were implemented in this study to build predictive models for the Iris dataset. The purpose of using multiple models is to compare different learning approaches and evaluate their performance under the same dataset conditions. Each model was trained using the same feature set (sepal length, sepal width, petal length, petal width) and the same target variable after preprocessing. The models chosen include Decision Tree, Random Forest, and Logistic Regression, which are commonly used algorithms in supervised machine learning. By applying and comparing these three models, we can evaluate their performance and determine which algorithm provides the most accurate predictions for this dataset.
</p>

### **a) Decision Tree**
<p align="justify">
The Decision Tree is a tree-based machine learning model that makes predictions by recursively splitting the dataset based on feature values. At each split, the model selects the feature that best separates the classes, forming a tree-like structure of decision rules. This model is easy to interpret because the decision process can be visualized as a flow of conditions from root to leaf nodes. The code to create the Decison Tree Model is shown below:
</p>

In [42]:
# Decision Tree
# Import the DecisionTreeClassifier from the PySpark ML classification module
from pyspark.ml.classification import DecisionTreeClassifier

# Initialize the Decision Tree model, specifying the input features and the target label
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)

<p align="justify">
This line of code creates a Decision Tree classification model using PySpark’s machine learning library. The "DecisionTreeClassifier" is imported from the ML classification module, and then initialized as "dt". During initialization, the "featuresCol="features"" parameter specifies the input feature vector used for training, while "labelCol="label"" defines the target variable that the model will learn to predict. This step prepares the model for training on the dataset.
</p>

### **b) Random Forest**
<p align="justify">
The Random Forest classifier is an ensemble learning method that improves upon the Decision Tree approach by building multiple decision trees and combining their outputs. Each tree is trained on a random subset of the data and features, and the final prediction is made based on majority voting across all trees. This approach reduces overfitting and improves generalization performance. The code to create the Random Forest classification model is shown below:
</p>

In [43]:
# Random Forest
# Import the RandomForestClassifier from the PySpark ML classification module
from pyspark.ml.classification import RandomForestClassifier

# Initialize the Random Forest model, specifying the input features and the target label
rf = RandomForestClassifier(featuresCol="features", labelCol="label" , seed=42)

<p align="justify">
Same as Decision Tree code previously, The lines of code above initialize a Random Forest classification model using PySpark’s machine learning library. The "RandomForestClassifier" is imported from the ML classification module and assigned to the variable "rf". The parameter "featuresCol="features"" specifies the input feature vector used for training, while "labelCol="label"" defines the target variable that the model will learn to predict. This setup prepares the Random Forest model, which will later be trained using multiple decision trees to improve prediction accuracy and reduce overfitting.
</p>

### **c) Logistic Regression**
<p align="justify">
Logistic Regression is a linear classification algorithm that models the probability of different classes using a logistic function. Although it is originally designed for binary classification, it can be extended to handle multiclass problems such as the Iris dataset. The model assumes a linear relationship between the input features and the log-odds of the target classes. It is simple, efficient, and serves as a strong baseline model for comparison with more complex algorithms.
</p>

In [44]:
# Logistic regression
# Import the LogisticRegression from the PySpark ML classification module
from pyspark.ml.classification import LogisticRegression

# Initialize the Logistic Regression model, specifying the input features and the target label
lr = LogisticRegression(featuresCol="features", labelCol="label")

<p align="justify">
The lines of code above initialize a Logistic Regression model using PySpark’s machine learning library. The "LogisticRegression" class is imported from the ML classification module and assigned to the variable lr. The parameter "featuresCol="features"" specifies the input feature vector used for training, while "labelCol="label"" defines the target variable that the model will predict. This setup prepares the Logistic Regression model, which will learn the relationship between the input features and the target classes for multiclass classification of the Iris dataset.
</p>

## **HYPERPARAMETER TUNING**
<p align="justify">
Hyperparameter tuning was performed to improve model performance and ensure that each algorithm was optimised for the dataset. This process involved using Grid Search in combination with 5-fold Cross-Validation, which allows multiple combinations of parameters to be tested systematically. Grid Search evaluates different hyperparameter settings, while cross-validation ensures that each configuration is assessed fairly across different subsets of the data. This approach helps to identify the best-performing model settings while reducing the risk of overfitting.
</p>
<p align="justify">
Hyperparameter tuning is essential because it helps improve model accuracy and generalisation by identifying the most suitable parameter settings. Without tuning, models may either underfit or overfit the data, leading to poor performance on unseen samples. By combining grid search with cross-validation, this ensure that each model is evaluated thoroughly and trained under optimal configurations, resulting in more reliable and robust predictions.
</p>

### **a) Decision Tree**


In [45]:
# Import ParamGridBuilder to construct a grid of parameters for tuning, and CrossValidator for k-fold evaluation
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Import MulticlassClassificationEvaluator to measure model performance (e.g., accuracy, F1-score)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [46]:
# Initialize the Parameter Grid Builder for the Decision Tree model
# Add maxDepth parameter with values 2, 5, and 10 to the grid
# Add minInstancesPerNode parameter with values 1, 2, and 4 to the grid
paramGrid_dt = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [2, 5, 10]) \
    .addGrid(dt.minInstancesPerNode, [1, 2, 4]) \
    .build()

# Initialize the evaluator using 'accuracy' as the primary performance metric
evaluator = MulticlassClassificationEvaluator(metricName="accuracy")

# Initialize the CrossValidator object
cv_dt = CrossValidator(
    estimator=dt, # Set the DecisionTreeClassifier as the base estimator
    estimatorParamMaps=paramGrid_dt, # Provide the parameter grid defined above
    evaluator=evaluator, # Provide the evaluator to measure performance
    numFolds=5 # Specify 5-fold cross-validation
)

# Train the model by fitting the CrossValidator to the training data
# This will find and return the best model based on the accuracy metric
model_dt = cv_dt.fit(train_df)

<p align="justify">
This block of code performs automated hyperparameter tuning for the Decision Tree model using a Grid Search and Cross-Validation in PySpark. The "ParamGridBuilder" is used to define a range of hyperparameter values to test for the Decision Tree model, including different values of "maxDepth" (tree depth) and  "minInstancesPerNode" (minimum samples required to split a node). The "build()" function finalizes the grid of parameter combinations that will be evaluated. Next, a "MulticlassClassificationEvaluator()" is created with accuracy as the evaluation metric to measure model performance. It then sets up a 5-fold Cross-Validator, which splits the training data into five parts to ensure the model's performance is robust and not due to chance. Finally, "cv_dt.fit(train_df)" trains the model on the training data and tests all combinations of hyperparameters in the grid. The best-performing model is selected based on the highest cross-validation accuracy, which is then stored as "model_dt".
</p>



### **b) Random Forest**


In [47]:
# Initialize the Parameter Grid Builder for the Random Forest model
# Add numTrees parameter with values 10, 20, and 30 (number of trees in the forest)
# Add maxDepth parameter with values 2, 5, and 10 to control tree growth
paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20, 30]) \
    .addGrid(rf.maxDepth, [2, 5, 10]) \
    .build()

# Initialize the CrossValidator for Random Forest
cv_rf = CrossValidator(
    estimator=rf, # Set the Random Forest classifier as the base model
    estimatorParamMaps=paramGrid_rf, # Provide the parameter grid defined above
    evaluator=evaluator, # The evaluator (accuracy) defined earlier
    numFolds=5 # 5-fold cross-validation
)

# Train the model and find the best parameter combination
model_rf = cv_rf.fit(train_df)

<p align="justify">
The codes above perform hyperparameter tuning for the Random Forest model using Grid Search and Cross-Validation. It begins by using "ParamGridBuilder" to define a search space for "numTrees" (the number of decision trees in the ensemble) and "maxDepth" (the depth of each tree). A "CrossValidator" is then initialized with a 5-fold strategy, which ensures the model's performance is validated across different subsets of the training data to improve reliability. Finally, the "fit()" method executes the search, training multiple versions of the model and selecting the optimal combination of parameters based on the accuracy metric to produce the final "model_rf".
</p>

### **c) Logistic Regression**

In [48]:
# Initialize the Parameter Grid Builder for the Logistic Regression model
# Add regParam (regularization) and elasticNetParam (mixing parameter) to the grid
paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Initialize the CrossValidator for Logistic Regression
cv_lr = CrossValidator(
    estimator=lr, # Set the Logistic Regression classifier as the base model
    estimatorParamMaps=paramGrid_lr, # Provide the parameter grid defined above
    evaluator=evaluator, # The evaluator (accuracy) defined earlier
    numFolds=5 # 5-fold cross-validation
)

# Train the model and find the best parameter combination
model_lr = cv_lr.fit(train_df)

<p align="justify">
"ParamGridBuilder" is used to define different combinations of hyperparameters to be tested. The parameters include "regParam", which controls the strength of regularization to prevent overfitting, and "elasticNetParam", which determines the balance between L1 (Lasso) and L2 (Ridge) regularization. By testing multiple values for these parameters, the model can identify the best configuration for improved performance.
</p>
<p align="justify">
Next, "CrossValidator" is created to evaluate the Logistic Regression model using 5-fold cross-validation. "cv_lr.fit(train_df)" trains the Logistic Regression model on the training dataset and evaluates all combinations in the parameter grid. The best model is selected based on the highest cross-validation accuracy, ensuring optimal performance and better generalisation on unseen data.
</p>

## **MODEL EVALUATION**
<p align="justify">
The performance of the classification models was evaluated using several standard evaluation metrics to ensure a comprehensive assessment of their predictive ability. These metrics provide different perspectives on how well each model performs, particularly in correctly classifying the Iris species based on the input features.
</p>
<p align="justify">
Accuracy was used as the primary evaluation metric and measures the overall proportion of correctly predicted instances out of all predictions made (evidentlyai.com). It provides a general indication of model performance; however, it may not always fully reflect performance in cases where class distribution is imbalanced.
</p>
<p align="justify">
Precision evaluates the correctness of positive predictions made by the model (evidentlyai.com). It indicates how many of the predicted positive cases are actually correct, making it useful for understanding the model’s reliability in classification decisions.
</p>
<p align="justify">
Recall measures the model’s ability to correctly identify all actual positive cases (evidentlyai.com). It shows how well the model captures all relevant instances of a particular class, which is important for ensuring that no class is overlooked.
</p>
<p align="justify">
F1-score is the harmonic mean of precision and recall, providing a balanced evaluation of both metrics (medium.com). It is particularly useful when there is a need to balance false positives and false negatives, offering a more comprehensive view of model performance than accuracy alone.
</p>
<p align="justify">
Overall, while accuracy provides a general measure of correctness, the F1-score offers a more reliable evaluation for classification tasks as it considers both precision and recall. This makes it a more informative metric when comparing the performance of different machine learning models. For this study, accuracy and F1-score metrics will be used to evaluate the models' performance.
</p>

####**Reference:**
(evidentlyai.com) from https://www.evidentlyai.com/classification-metrics/accuracy-precision-recall

(medium.com) from https://medium.com/@piyushkashyap045/understanding-precision-recall-and-f1-score-metrics-ea219b908093

### **a) Decision Tree**

In [49]:
# Generate predictions for the test dataset using the best Decision Tree model
pred_dt = model_dt.transform(test_df)

# Initialize evaluators for both 'accuracy' and 'f1' metrics
evaluator_acc = MulticlassClassificationEvaluator(metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(metricName="f1")

# Evaluate the model performance using the accuracy and F1-score metrics
accuracy_dt = evaluator_acc.evaluate(pred_dt)
f1_dt = evaluator_f1.evaluate(pred_dt)

# Output the final performance results for Decision Tree model
print("Decision Tree Accuracy:", accuracy_dt)
print("Decision Tree F1:", f1_dt)

Decision Tree Accuracy: 1.0
Decision Tree F1: 1.0


<p align="justify">
This code evaluates the performance of the trained Decision Tree model on the test dataset. "transform()" function is used to generate predictions "(pred_dt)" by applying the trained model to the unseen test data. Next, two evaluation metrics; accuracy and F1-score are defined using "MulticlassClassificationEvaluator". These metrics measure the model’s overall correctness and the balance between precision and recall, respectively. The "evaluate()" function is then used to compute the accuracy and F1-score based on the predicted results. Finally, the performance values are printed to display the Decision Tree model’s classification effectiveness on the testing dataset.
</p>

### **b) Random Forest**

In [50]:
# Generate predictions for the test dataset using the best Random Forest model
pred_rf = model_rf.transform(test_df)

# Evaluate the model performance using the accuracy and F1-score metrics
accuracy_rf = evaluator_acc.evaluate(pred_rf)
f1_rf = evaluator_f1.evaluate(pred_rf)

# Output the final performance results for Random Forest model
print("Random Forest Accuracy:", accuracy_rf)
print("Random Forest F1:", f1_rf)

Random Forest Accuracy: 1.0
Random Forest F1: 1.0


<p align="justify">
The code aboves apply the same method for Random Forest as per the Decision Tree. This is again the same line of codes adjusted accordingly for Logistic Regression as per the below code:
</p>

### **c) Logistic Regression**

In [51]:
# Generate predictions for the test dataset using the best Logistic Regression model
pred_lr = model_lr.transform(test_df)

# Evaluate the model performance using the accuracy and F1-score metrics
accuracy_lr = evaluator_acc.evaluate(pred_lr)
f1_lr = evaluator_f1.evaluate(pred_lr)

# Output the final performance results for Logistic Regression
print("Logistic Regression Accuracy:", accuracy_lr)
print("Logistic Regression F1:", f1_lr)

Logistic Regression Accuracy: 1.0
Logistic Regression F1: 1.0


## **PREDICTION**
<p align="justify">
In this step, the optimised machine learning model is used to generate predictions on the testing dataset. After training and tuning the model using cross-validation, it is applied to unseen data to evaluate its real-world performance. The transform() function is used to produce predicted class labels based on the input features, allowing comparison between the predicted values and the actual labels. This step is essential for assessing how well the model generalises to new data.
</p>

### **a) Decision Tree**

In [52]:
# Generate predictions using the trained Decision Tree model
dt_predictions = model_dt.transform(test_df)

# Show sample predictions
dt_predictions.select("features", "label", "prediction").show()

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.6,3.6,1.0,0.2]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.1,1.5,0.1]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.1,3.5,1.4,0.2]|  0.0|       0.0|
|[5.3,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.0,4.5,1.5]|  1.0|       1.0|
|[5.4,3.4,1.5,0.4]|  0.0|       0.0|
|[5.4,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.9,1.7,0.4]|  0.0|       0.0|
|[5.5,2.5,4.0,1.3]|  1.0|       1.0|
|[5.6,2.9,3.6,1.3]|  1.0|       1.0|
|[5.7,2.9,4.2,1.3]|  1.0|       1.0|
|[5.8,2.7,5.1,1.9]|  2.0|       2.0|
|[6.3,2.5,4.9,1.5]|  1.0|       1.0|
|[6.4,3.1,5.5,1.8]|  2.0|       2.0|
|[6.5,3.0,5.2,2.0]|  2.0|       2.0|
+-----------------+-----+----------+
only showing top 20 rows


<p align="justify">
The code above applies the trained Decision Tree model to the testing dataset using the "transform()" function. It generates predictions for unseen data by adding a new column called "prediction", which represents the predicted class label for each observation. These predictions can then be compared with the actual labels to evaluate model performance.
</p>

### **b) Random Forest**

In [53]:
# Generate predictions using the trained Random Forest model
rf_predictions = model_rf.transform(test_df)

# Show sample predictions
rf_predictions.select("features", "label", "prediction").show()

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.6,3.6,1.0,0.2]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.1,1.5,0.1]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.1,3.5,1.4,0.2]|  0.0|       0.0|
|[5.3,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.0,4.5,1.5]|  1.0|       1.0|
|[5.4,3.4,1.5,0.4]|  0.0|       0.0|
|[5.4,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.9,1.7,0.4]|  0.0|       0.0|
|[5.5,2.5,4.0,1.3]|  1.0|       1.0|
|[5.6,2.9,3.6,1.3]|  1.0|       1.0|
|[5.7,2.9,4.2,1.3]|  1.0|       1.0|
|[5.8,2.7,5.1,1.9]|  2.0|       2.0|
|[6.3,2.5,4.9,1.5]|  1.0|       1.0|
|[6.4,3.1,5.5,1.8]|  2.0|       2.0|
|[6.5,3.0,5.2,2.0]|  2.0|       2.0|
+-----------------+-----+----------+
only showing top 20 rows


<p align="justify">
Above code uses the trained Random Forest model to make predictions on the test dataset. The "transform()" function applies the ensemble model to unseen data and outputs predicted class labels. These predictions are stored in a new column called "prediction", which is used for further evaluation of model accuracy and performance.
</p>

### **c) Logistic Regression**

In [54]:
# Generate predictions using the trained Logistic Regression model
lr_predictions = model_lr.transform(test_df)

# Show sample predictions
lr_predictions.select("features", "label", "prediction").show()

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.6,3.6,1.0,0.2]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.1,1.5,0.1]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.1,3.5,1.4,0.2]|  0.0|       0.0|
|[5.3,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.0,4.5,1.5]|  1.0|       1.0|
|[5.4,3.4,1.5,0.4]|  0.0|       0.0|
|[5.4,3.7,1.5,0.2]|  0.0|       0.0|
|[5.4,3.9,1.7,0.4]|  0.0|       0.0|
|[5.5,2.5,4.0,1.3]|  1.0|       1.0|
|[5.6,2.9,3.6,1.3]|  1.0|       1.0|
|[5.7,2.9,4.2,1.3]|  1.0|       1.0|
|[5.8,2.7,5.1,1.9]|  2.0|       2.0|
|[6.3,2.5,4.9,1.5]|  1.0|       1.0|
|[6.4,3.1,5.5,1.8]|  2.0|       2.0|
|[6.5,3.0,5.2,2.0]|  2.0|       2.0|
+-----------------+-----+----------+
only showing top 20 rows


<p align="justify">
Based on the code above, it applies the trained Logistic Regression model to the test dataset to generate predictions. The model outputs a probability-based classification result, which is converted into a final predicted label stored in the "prediction" column. These results are then compared with the actual labels to assess model performance.
</p>

## **COMPARATIVE ANALYSIS**

<p align="justify">
For this section, all three classification models; Decision Tree, Random Forest, and Logistic Regression are evaluated using standard performance metrics, including accuracy and F1-score side by side. These metrics provide a comprehensive assessment of each model’s predictive performance on the testing dataset. By comparing multiple evaluation measures rather than relying solely on accuracy, a more balanced and reliable understanding of model effectiveness is obtained. This allows for an objective comparison of how well each model performs in classifying the Iris dataset. The code is shown below:
</p>

In [55]:
# Import the MulticlassClassificationEvaluator class for model assessment
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Define the evaluator for the 'accuracy' metric using specific label and prediction columns
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Define the evaluator for the 'f1' score metric using specific label and prediction columns
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# ---- Decision Tree metrics ----
# Calculate accuracy for the Decision Tree predictions
dt_acc = evaluator_acc.evaluate(dt_predictions)
# Calculate F1-score for the Decision Tree predictions
dt_f1 = evaluator_f1.evaluate(dt_predictions)

# ---- Random Forest metrics ----
# Calculate accuracy for the Random Forest predictions
rf_acc = evaluator_acc.evaluate(rf_predictions)
# Calculate F1-score for the Random Forest predictions
rf_f1 = evaluator_f1.evaluate(rf_predictions)

# ---- Logistic Regression metrics ----
# Calculate accuracy for the Logistic Regression predictions
lr_acc = evaluator_acc.evaluate(lr_predictions)
# Calculate F1-score for the Logistic Regression predictions
lr_f1 = evaluator_f1.evaluate(lr_predictions)

<p align="justify">
A comparison table is created to summarise the performance of all three classification models: Decision Tree, Random Forest, and Logistic Regression. The table presents key evaluation metrics, including accuracy and and F1-score, in a structured format. This allows for an easy and direct comparison of model performance, making it clearer to identify which model performs best across different evaluation criteria. Below code generates the comparison table:
</p>

In [56]:
import pandas as pd

# Create a list of lists containing the model names and their corresponding performance metrics
data = [
    ["Decision Tree", dt_acc, dt_f1],
    ["Random Forest", rf_acc, rf_f1],
    ["Logistic Regression", lr_acc, lr_f1]
]

# Define the column names for the pandas DataFrame
columns = ["Model", "Accuracy", "F1-score"]

# Initialize a pandas DataFrame using the data and column names defined above
comparison_df = pd.DataFrame(data, columns=columns)

# Display the resulting comparison DataFrame
display(comparison_df)

,Model,Accuracy,F1-score
0,Decision Tree,1.0,1.0
1,Random Forest,1.0,1.0
2,Logistic Regression,1.0,1.0


<p align="justify">
Reorder the table to show the best performing model from the best to the worst. The code is shown below:
</p>

In [57]:
# Sort the comparison table by F1-score in descending order to identify the best performing models
best_model = comparison_df.sort_values(by="F1-score", ascending=False)

print("Best Performing Model Summary:")

# Display the DataFrame with 4 decimal places using pandas styling for a better visual representation
display(best_model.round(4).style.hide(axis='index'))

Best Performing Model Summary:


Model,Accuracy,F1-score
Decision Tree,1.000000,1.000000
Random Forest,1.000000,1.000000
Logistic Regression,1.000000,1.000000
